In [ ]:
# @title [このノートブックをリポジトリで見る](https://github.com/ec22s/colab-ikinari-python/blob/main/chapter-5/colab-chapter-5.ipynb)

In [ ]:
# @title (1) p.142-146 本のままではColabで動かない（PCのカメラで写真撮影）

import cv2

# カメラの初期化
cap = cv2.VideoCapture(0)

# 静止画の取得
ret, frame = cap.read()

# 画像を保存
if ret:
  cv2.imwrite('img.jpg', frame)

# 解放処理
cap.release()

# 以上がp.146にあるコード5-1-2
# Colabでは、9行目のcap.read()で静止画を取得できない
# そこで学習会用に cv2.VideoCapture(0) を代替する独自クラス ColabCap を作った
# 次セルからそれを使う

In [ ]:
# @title (2) Chapter 5用の独自クラス・関数をまとめて準備

import requests

basenames = ["ColabCap", "VideoWriter", "play_video"]

repository = "https://github.com/ec22s/colab-ikinari-python"
path = "raw/refs/heads/main/chapter-5"
for basename in basenames:
  url = f"{repository}/{path}/{basename}.py"
  exec(requests.get(url, allow_redirects=True).content.decode("utf-8"))

# 独自クラス ColabCap に加え、動画用の独自関数2つもこの際に準備
# 動画用はChapter 5後半とChapter 6で使う
# リポジトリでの解説 → https://github.com/ec22s/colab-ikinari-python/blob/main/util/load-py.md

In [ ]:
#@title (3) 独自クラス ColabCap のテスト。本 p.146 (5-1-2) と同様、静止画を撮影・保存

# 初回実行時はたいていカメラ利用許可がまだなく、エラーになる
# 2回目の実行でカメラ利用許可の確認ダイアログが出たらOKする
# 動作確認済 2026.6.18 Chrome 148.0, Firefox 151.0

# カメラの初期化
# cap = cv2.VideoCapture(0) # 本
cap = ColabCap() # Colab版

# 静止画の取得
ret, frame = cap.read()

# 画像を保存
if ret:
  cv2.imwrite('img.jpg', frame)

# 解放処理
cap.release()

In [ ]:
#@title (4) 静止画撮影に加え、Colabで画像を表示。本 p.148 (5-1-3) と同様

# 初回実行時はたいていカメラ利用許可がまだなく、エラーになる
# 2回目の実行でカメラ利用許可の確認ダイアログが出たらOKする
# 動作確認済 2026.6.18 Chrome 148.0, Firefox 151.0

# 画像表示はColab用ライブラリを使う
from google.colab.patches import cv2_imshow

# カメラの初期化
cap = ColabCap()

# 静止画の取得
ret, frame = cap.read()

# 画像を保存
if ret:
  cv2.imwrite('img.jpg', frame)

# 解放処理
cap.release()

# 結果表示
cv2_imshow(frame)


In [ ]:
#@title (5) 動画撮影・保存：本 p.149〜153 (5-2) と同様に1コマずつ撮影

# この方法では滑らかな動画にならない. 1コマずつブラウザで取得しColabへ送るため
# 動画表示も遅いので省略

# 初回実行時はたいていカメラ利用許可がまだなく、エラーになる
# 2回目の実行でカメラ利用許可の確認ダイアログが出たらOKする
# 動作確認済 2026.6.18 Chrome 148.0, Firefox 151.0

import cv2
import time

# カメラの初期化
cap = ColabCap()

# 撮影条件
frame_rate = 10 # Colabでは10fps位が限界. sleepなしで限界を確かめ設定するのが現実的
duration = 10
interval = 1 / frame_rate
frame_count = int(duration / interval)

# 動画保存条件
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('movie.mp4', fourcc, frame_rate, (640, 480))

# 動画撮影
for i in range(frame_count):
  ret, frame = cap.read()
  if not ret:
    break
  out.write(frame)

  # 再生すると極端に遅くなる (出力欄をつど消去するため)
  # cv2_imshow(frame)
  # output.clear()

  # time.sleep(interval)
  # 1コマずつのスリープ、Colabでは不要（撮影自体が遅いため）

cap.release()
out.release()

In [ ]:
#@title (6) タイムラプス動画撮影・保存：本 p.154〜157 (5-3) に近い結果を得る

# 初回実行時はたいていカメラ利用許可がまだなく、エラーになる
# 2回目の実行でカメラ利用許可の確認ダイアログが出たらOKする
# 動作確認済 2026.6.18 Chrome 148.0, Firefox 151.0

import cv2
import time
from google.colab.patches import cv2_imshow
from google.colab import output

# カメラの初期化
cap = ColabCap()

# 撮影条件
frame_rate = 10
duration = 60
interval = 2
frame_count = int(duration / interval)

# 動画保存条件
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('movie_timelapse.mp4', fourcc, frame_rate, (640, 480))

# 動画撮影
for i in range(frame_count):
  ret, frame = cap.read()
  if not ret:
    break
  out.write(frame)

  cv2_imshow(frame)

  time.sleep(interval)
  # 実際はintervalより少し長めに待機してしまう. ブラウザの処理と通信が入るため
  output.clear()

cap.release()
out.release()

In [ ]:
# @title (7) 画像処理の準備：本 p.159 (5-4-1) と同様に画像を表示

# Colabにまだ画像がなければ (4) を実行し、こちらは不要
# 画像のファイル名が img.jpg 以外なら10行目を変更する

import cv2
from google.colab.patches import cv2_imshow

# 画像ファイルの読み込み
img = cv2.imread('img.jpg')

# 画像の表示
cv2_imshow(img)

In [ ]:
# @title (8) 画像をセピア色に変換：本 p.160〜161 (5-4-2) と同様

from google.colab.patches import cv2_imshow
import cv2
import numpy as np

def apply_color_tone(img):
  """画像に色効果を適用する関数"""

  # セピア調にするカラー変換行列
  # 注：0.xxxの先頭ゼロは略せる 工学系のよくある記法
  # 注：インデント浅くした

  sepia_filter = np.array([
    [.272, .534, .131],
    [.349, .686, .168],
    [.393, .769, .189]
  ])
  applied_img = cv2.transform(img, sepia_filter)

  # 値を0〜255の範囲に変更
  applied_img = np.clip(applied_img, 0, 255).astype(np.uint8)

  return applied_img

# 画像ファイルの読み込み
img = cv2.imread('img.jpg')

# 画像処理を実行
applied_img = apply_color_tone(img)

# 画像の保存
# 表示より先にした. imwriteの戻り値True or Falseを表示しないため
cv2.imwrite('img_out.jpg', applied_img)

# 画像の表示
cv2_imshow(applied_img)

In [ ]:
# @title (9) 好きな色変換を試す：本 p.163 (5-4-3) と同様

from google.colab.patches import cv2_imshow
import cv2
import numpy as np

def apply_color_tone(img):
  """画像に色効果を適用する関数"""

  # セピア以外の変換の例 本p.166 ⑥全体的に明るくする
  filter = np.array([
    [3, 0, 0],
    [0, 3, 0],
    [0, 0, 3]
  ])
  applied_img = cv2.transform(img, filter)

  # 値を0〜255の範囲に変更
  applied_img = np.clip(applied_img, 0, 255).astype(np.uint8)

  return applied_img

# 画像ファイルの読み込み
img = cv2.imread('img.jpg')

# 画像処理を実行
applied_img = apply_color_tone(img)

# 画像の保存
cv2.imwrite('img_out.jpg', applied_img)

# 画像の表示
cv2_imshow(applied_img)

In [ ]:
# @title (10) 画像のエッジを強調：本 p.166〜167 (5-4-4) と同様

from google.colab.patches import cv2_imshow
import cv2

def apply_edges(img):
  """エッジを検出して元画像に重ね描きする関数"""

  # グレースケールに変換
  gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

  # エッジを検出
  edges = cv2.Canny(gray, 100, 200)

  # エッジを黒色で描画
  img[edges == 255] = (0, 0, 0)

  return img

# 画像ファイルの読み込み
img = cv2.imread('img.jpg')

# 画像処理を実行
applied_img = apply_edges(img)

# 画像の表示
cv2_imshow(applied_img)

In [ ]:
# @title (11) 画像のエッジを太くする：本 p.169 (5-4-5) と同様

import numpy as np
from google.colab.patches import cv2_imshow
import cv2

def apply_edges(img):
  gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
  edges = cv2.Canny(gray, 100, 200)

  # エッジを太くする
  kernel = np.ones((3, 3), np.uint8)
  edges_dilated = cv2.dilate(edges, kernel, iterations=1)

  # エッジを黒色で描画
  img[edges_dilated == 255] = (0, 0, 0)

  return img

# 画像ファイルの読み込み
img = cv2.imread('img.jpg')

# 画像処理を実行
applied_img = apply_edges(img)

# 画像の表示
cv2_imshow(applied_img)

In [ ]:
# @title (12) 画像をぼかす：本 p.170 (5-4-6) と同様

from google.colab.patches import cv2_imshow
import cv2

def apply_blur(img):
  """画像全体にぼかし効果を追加する関数"""
  kernel = (15, 15)
  return cv2.GaussianBlur(img, kernel, 0)

# 画像ファイルの読み込み
img = cv2.imread('img.jpg')

# 画像処理を実行
applied_img = apply_blur(img)

# 画像の表示
cv2_imshow(applied_img)

In [ ]:
# @title (13) 動画再生の前段：本の方法で作成した動画は、そのままではColabで再生できない

# セル(5)または(6)を実行した結果のmovie.mp4がなければ再実行してから

from IPython.display import display
from base64 import b64encode

mp4 = open('movie.mp4', "rb").read()
b64 = b64encode(mp4).decode()

display(Javascript(f"""
  const video = document.createElement("video");
  video.autoplay = true;
  video.controls = true;
  const source = document.createElement("source");
  source.src = "data:video/mp4;base64,{b64}";
  video.appendChild(source);
  document.querySelector("#output-area").appendChild(video);
"""));
# GitHubで正しく表示できるようHTMLタグを使わずJavaScriptで書いた

In [ ]:
# @title (14) 動画再生：学習会の独自関数 play_video を使えば再生できる

# セル(2)関数準備がまだなら、先に実行
# セル(5)or(6)を実行した結果のmovie.mp4がなければ、先に実行

play_video("movie.mp4", 320)

# 第2引数で再生時の縦の長さを指定できる

In [ ]:
# @title (15) 動画の撮影・保存・再生を簡単に実行：独自関数2つを利用

out_file = "movie.mp4"
duration = 10
frame_rate = 30

if VideoWriter(out_file, duration, frame_rate, (640, 480)):
  play_video(out_file, 320)

In [ ]:
# @title (16) 動画の色変換・ぼかし：本 p.172〜173 (5-5-1) と同様に。最後に自動再生

import cv2
import numpy as np

def apply_color_tone(img):
  """画像に色効果を適用する関数"""

  # セピア以外の変換の例 本p.166 ⑥全体的に明るくする
  filter = np.array([
    [3, 0, 0],
    [0, 3, 0],
    [0, 0, 3]
  ])
  applied_img = cv2.transform(img, filter)

  # 値を0〜255の範囲に変更
  applied_img = np.clip(applied_img, 0, 255).astype(np.uint8)

  return applied_img

def apply_blur(img):
  """画像全体にぼかし効果を追加する関数"""
  kernel = (15, 15)
  return cv2.GaussianBlur(img, kernel, 0)

# 動画ファイルの読み込み
cap = cv2.VideoCapture("movie.mp4")

# フレームレートの取得
frame_rate = cap.get(cv2.CAP_PROP_FPS)

# 注：本の55行目 interval はどこからも呼ばれず不要

# 動画の幅と高さを取得
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# 動画保存条件
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter("movie_1_edited.mp4", fourcc, frame_rate, (w, h))

# 画像処理
while cap.isOpened():
  ret, img = cap.read()
  if not ret:
    break
  applied_img = apply_color_tone(img)
  out.write(apply_blur(applied_img))

# ファイルの解放
cap.release()
out.release()

play_video("movie_1_edited.mp4", 240)

In [ ]:
#@title (17) 動画の色変換とエッジ強調：本 p.175 (5-5-2) と同様に。最後に自動再生

import cv2
import numpy as np

def apply_color_tone(img):
  filter = np.array([
    [5, 0, 0],
    [0, 5, 0],
    [0, 0, 5]
  ])
  applied_img = cv2.transform(img, filter)

  # 値を0〜255の範囲に変更
  return np.clip(applied_img, 0, 255).astype(np.uint8)

def apply_edges(img):
  gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
  edges = cv2.Canny(gray, 100, 200)

  # エッジを太くする
  kernel = np.ones((3, 3), np.uint8)
  edges_dilated = cv2.dilate(edges, kernel, iterations=1)

  # エッジを黒色で描画
  img[edges_dilated == 255] = (0, 0, 0)

  return img

# 動画ファイルの読み込み
cap = cv2.VideoCapture("movie.mp4")

# フレームレートの取得
frame_rate = cap.get(cv2.CAP_PROP_FPS)

# 動画の幅と高さを取得
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# 動画保存条件
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter("movie_2_edited.mp4", fourcc, frame_rate, (w, h))

# 画像処理
while cap.isOpened():
  ret, img = cap.read()
  if not ret:
    break

  applied_image = apply_color_tone(img)
  applied_image = apply_edges(applied_image)
  out.write(applied_image)

# ファイルの解放
cap.release()
out.release()

play_video("movie_2_edited.mp4", 240)

In [ ]:
#@title (18) Chapter 5 最後：動画撮影〜色変換とエッジ強調〜自動再生をまとめて

from IPython.display import HTML
from base64 import b64encode
import cv2
import numpy as np

def apply_color_tone(img):
  filter = np.array([
    [5, 0, 0],
    [0, 5, 0],
    [0, 0, 5]
  ])
  applied_img = cv2.transform(img, filter)

  # 値を0〜255の範囲に変更
  return np.clip(applied_img, 0, 255).astype(np.uint8)

def apply_edges(img):
  gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
  edges = cv2.Canny(gray, 100, 200)

  # エッジを太くする
  kernel = np.ones((3, 3), np.uint8)
  edges_dilated = cv2.dilate(edges, kernel, iterations=1)

  # エッジを黒色で描画
  img[edges_dilated == 255] = (0, 0, 0)

  return img

# PCカメラから動画ファイルを保存し読み込み
frame_rate = 30
duration = 10
VideoWriter("movie_3.mp4", duration, frame_rate, (640, 480))
cap = cv2.VideoCapture("movie_3.mp4")

# フレームレートの取得
frame_rate = cap.get(cv2.CAP_PROP_FPS)
# 動画の幅と高さを取得
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# 編集した動画の保存条件
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter("movie_3_edited.mp4", fourcc, frame_rate, (w, h))

# 動画編集
while cap.isOpened():
  ret, img = cap.read()
  if not ret:
    break

  # 画像処理を実行
  applied_image = apply_color_tone(img)
  applied_image = apply_edges(applied_image)
  out.write(applied_image)

# ファイルの解放
cap.release()
out.release()

play_video("movie_3_edited.mp4", 240)